In [1]:
# Saving country level OSDMA8 for all ensemble members

In [2]:
import os
import xarray as xr
import numpy as np
from utils.utils import lat_weighted_mean
from utils.utils import get_scenario_config

In [3]:
# === Path config ===
MASK_DIR = "/glade/work/awells/air_quality/BMR/masks/country/"

in_file = "GBD_Country_Masks_0.10_newlabels.nc"
in_path = os.path.join(MASK_DIR, in_file)
country_mask = xr.open_dataarray(in_path)

In [4]:
# === Scenario and path config ===
# Set to whatever scenario and model you want
# Function returns error if not recognised
model = "UKESM1"
scenario = "G6-1.5K"

config = get_scenario_config(model, scenario)
ensemble_members = config["ensemble_members"]
years = config["years"]


OSDMA8_DIR = f"/glade/work/awells/air_quality/{model}/ozone/OSDMA8_BC/"

# === Main loop ===
ensembles = []
for ens_num in ensemble_members:
    print(f"Processing {scenario}, Ensemble {ens_num:02d}")
    dates = f"{years.start}-{years.stop}"

    in_file = f"OSDMA8_BC_{model}_{scenario}_{ens_num:02d}_{dates}.nc"
    in_path = os.path.join(OSDMA8_DIR, in_file)

    if not os.path.exists(in_path):
        print(f"Missing: {in_path}")
        continue
    da = xr.open_dataarray(in_path)

    # Adjust indices to match (with small tolerance)
    # e.g., max 1e-7 km distance
    da_adjust = da.reindex_like(country_mask, method="nearest",
                                tolerance=1e-9)

    countries = []
    # Loop over countries and take the OSDMA8 mean for each country
    for i in range(len(country_mask["country"])):
        mask = country_mask.isel(country=i)
        # osdma8 of country
        o3_country = lat_weighted_mean(xr.where(mask == 1, da_adjust,
                                                np.nan))
        countries.append(o3_country.drop_vars("country", errors='ignore'))
    osdma8_country = xr.concat(
        countries,
        dim=xr.DataArray(
            country_mask["country"],
            dims="country",
            name="country"))

    ensembles.append(osdma8_country)

country_ens = xr.concat(ensembles,
                        dim=xr.DataArray(np.arange(1, len(ensembles)+1),
                                         dims="ensemble", name="ensemble"))

country_ens.attrs["description"] = ("Country level mean OSDMA8 - scripts "
                                    "by A.F. Wells (2025)")
country_ens.attrs["units"] = "ppb"
country_ens.attrs["scenario"] = scenario
country_ens.attrs["model"] = model

print(f"Saving country level OSDMA8 to {OSDMA8_DIR}")
out_file = f"OSDMA8_BC_Country_mean_{model}_{scenario}_{dates}.nc"
out_path = os.path.join(OSDMA8_DIR, out_file)
country_ens.to_netcdf(out_path)

Processing ARISE, Ensemble 01
Processing ARISE, Ensemble 02
Processing ARISE, Ensemble 03
Processing ARISE, Ensemble 04
Processing ARISE, Ensemble 05
Processing ARISE, Ensemble 06
Processing ARISE, Ensemble 07
Processing ARISE, Ensemble 08
Processing ARISE, Ensemble 09
Processing ARISE, Ensemble 10
Saving country level OSDMA8 to /glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/
Processing SSP245, Ensemble 01
Processing SSP245, Ensemble 02
Processing SSP245, Ensemble 03
Processing SSP245, Ensemble 04
Processing SSP245, Ensemble 05
Processing SSP245, Ensemble 06
Processing SSP245, Ensemble 07
Processing SSP245, Ensemble 08
Processing SSP245, Ensemble 09
Processing SSP245, Ensemble 10
Saving country level OSDMA8 to /glade/work/awells/air_quality/CESM/ozone/OSDMA8_BC/
